# Preprocessing Validation

Verify data changes at each preprocessing stage.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
plt.rcParams.update({'figure.figsize': (12, 6)})

## 1. Load Raw Data

In [ ]:
from src.preprocessing.loader import AISDKLoader

loader = AISDKLoader(data_root='./data', date_start='2024-03-01', date_end='2024-03-31')
df_raw = loader.load()
print(f'Raw records: {len(df_raw):,}')
df_raw.head()

## 2. Run Cleaner

In [ ]:
from src.preprocessing.cleaner import AISCleaner

cleaner = AISCleaner(output_path='outputs/processed/test_clean.parquet')
df_clean = cleaner.run(df_raw.copy())
print(f'Clean records: {len(df_clean):,}')
print(f'Report: {cleaner.report}')

In [ ]:
# Check cleaning report
for key, value in cleaner.report.items():
    pct = (value / len(df_raw)) * 100
    print(f'{key}: {value:,} records ({pct:.2f}%)')

## 3. Feature Engineering

In [ ]:
from src.preprocessing.feature_engineer import AISFeatureEngineer

engineer = AISFeatureEngineer()
df_features = engineer.run(df_clean.copy(), conflict_events_path='', ports_path='')
print(f'Feature records: {len(df_features):,}')
print(f'New columns: {[c for c in df_features.columns if c not in df_clean.columns]}')

In [ ]:
df_features.head(3)

## 4. Validate Key Features

In [ ]:
# Check speed_category
if 'speed_category' in df_features.columns:
    print(df_features['speed_category'].value_counts(dropna=False))

In [ ]:
# Check conflict zones
print(df_features['conflict_zone_name'].value_counts())
print(f"In conflict zone: {df_features['in_conflict_zone'].sum():,} records")

## 5. Aggregation

In [ ]:
from src.preprocessing.aggregator import AISAggregator

aggregator = AISAggregator(output_path='outputs/processed/test_agg.parquet')
df_agg = aggregator.run('outputs/processed/test_clean.parquet')
print(f'Aggregated rows: {len(df_agg):,}')
df_agg.head()

In [ ]:
# Check aggregated features
df_agg.describe()